# ViT-B/16 — Diabetic Retinopathy Grading

**EEEM068 Applied Machine Learning — University of Surrey**

Fine-tuning **ViT-B/16 pre-trained on ImageNet-21k** for 5-class DR grading.
Best configuration: two-phase fine-tuning with configurable imbalance handling (FocalLoss or WRS).

| Setting | Value |
|---|---|
| Architecture | `vit_base_patch16_224.augreg_in21k_ft_in1k` |
| Pre-training | ImageNet-21k → fine-tuned on ImageNet-1k |
| Imbalance handling | FocalLoss (alpha) **or** WeightedRandomSampler |
| Training | Two-phase: frozen head → full fine-tune |
| Scheduler | CosineAnnealingLR or LinearWarmup + CosineDecay |
| Metric | Quadratic Weighted Kappa (QWK) |

## 1. Imports

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import random
from pathlib import Path
from datetime import datetime

import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.metrics import (
    classification_report, cohen_kappa_score,
    confusion_matrix, f1_score, accuracy_score,
)
from sklearn.utils.class_weight import compute_class_weight

print(f'PyTorch : {torch.__version__}')
print(f'timm    : {timm.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')


## 2. Configuration

All paths, model settings, and Experiment 8 hyperparameters are defined here.
Edit the HP block before each run — nothing else needs changing.

**Set random seed for reproducibility and select the compute device.**

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
ROOT         = NOTEBOOK_DIR.parent
DATA_ROOT    = ROOT / 'data' / 'diabetic-retinopathy-detection'
TRAIN_DIR    = DATA_ROOT / 'train_proc'
VAL_DIR      = DATA_ROOT / 'val_proc'
TEST_LABELS_DIR = DATA_ROOT / 'test_labels_proc'
TRAIN_CSV    = DATA_ROOT / 'train_split.csv'
VAL_CSV      = DATA_ROOT / 'val_split.csv'
TEST_CSV     = DATA_ROOT / 'test_split.csv'

OUT_DIR      = ROOT / 'outputs' / 'vit_b16'
CKPT_DIR     = OUT_DIR / 'checkpoints'
FIGURES_DIR  = OUT_DIR / 'figures'
for d in (OUT_DIR, CKPT_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── Model & data constants ────────────────────────────────────────────────────
EXPERIMENT_NAME = 'vit_b16'
MODEL_NAME  = 'vit_base_patch16_224.augreg_in21k_ft_in1k'
SEED        = 42
IMAGE_SIZE  = 224
NUM_CLASSES = 5
CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
VIT_MEAN    = (0.5, 0.5, 0.5)
VIT_STD     = (0.5, 0.5, 0.5)
NUM_WORKERS = 4

# ── Hyperparameters — edit before each run ──────────────────────
USE_FOCAL_LOSS   = False  # True  → FocalLoss(alpha=class_weights) + shuffle
                          # False → CrossEntropyLoss + WeightedRandomSampler
USE_WARMUP_P1    = False  # Phase 1 scheduler: warmup then cosine decay
USE_WARMUP_P2    = True   # Phase 2 scheduler: warmup then cosine decay
WARMUP_EPOCHS = 3      # warmup length (per-phase, when enabled)
PHASE1_EPOCHS = 30
PHASE2_EPOCHS = 30
PHASE1_LR     = 1e-4   # head-only phase — stable convergence before full fine-tune
PHASE2_LR     = 5e-6   # full fine-tune — low to avoid catastrophic forgetting
BATCH_SIZE       = 16
WEIGHT_DECAY  = 3e-2
GRAD_CLIP     = 1.0
PATIENCE      = 5

print('Config ready.  Root:', ROOT)


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'
print('Device:', device, '| AMP:', use_amp)


**Define checkpoint paths. Re-run this cell after a kernel restart to resume evaluation.**

In [ ]:
# Run this cell alone after a kernel restart to re-enable evaluation.
CKPT_FINAL_P1 = ROOT / 'outputs' / 'vit_b16_final' / 'checkpoints' / 'vit_b16_final_phase1_best.pt'
CKPT_FINAL    = ROOT / 'outputs' / 'vit_b16_final' / 'checkpoints' / 'vit_b16_final_best.pt'

for label, path in [('Final Phase 1', CKPT_FINAL_P1), ('Final Best', CKPT_FINAL)]:
    status = 'saved' if path.exists() else 'not yet trained'
    print(f'  {label:<14}: {path.relative_to(ROOT)}  [{status}]')


## 3. Load Data

Reads the pre-split CSV files. Splits were created once and fixed — val/test are never resampled.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f'Train : {len(train_df):,} images  (dir: train/)')
print(f'Val   : {len(val_df):,} images   (dir: val/)')
print(f'Test  : {len(test_df):,} images   (dir: test_labels/)')
print()
print(f'{"Grade":<6} {"Name":<18} {"Train":>7} {"Val":>7} {"Test":>7}')
print('-' * 52)
for g, name in enumerate(CLASS_NAMES):
    t = (train_df['level'] == g).sum()
    v = (val_df['level']   == g).sum()
    s = (test_df['level']  == g).sum()
    print(f'  {g}    {name:<18} {t:>6,}  {v:>6,}  {s:>6,}')

## 4. Dataset & DataLoaders

Augmentations are applied only to the training set. Val and test use a clean resize + normalise.
The enhanced augmentation pipeline (flips, rotation, colour jitter, affine) matches Experiment 2.

### Class Weights & Sampler

Compute balanced inverse-frequency weights from the training labels. Used as FocalLoss alpha when `USE_FOCAL_LOSS=True`, or to drive `WeightedRandomSampler` when `USE_FOCAL_LOSS=False`.

In [ ]:
class DRDataset(Dataset):
    def __init__(self, df, data_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.data_dir  = Path(data_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(self.data_dir / f"{row['image']}.jpeg").convert('RGB')
        label = int(row['level'])
        if self.transform:
            image = self.transform(image)
        return image, label


# ViT requires symmetric [−1, 1] normalisation, not ImageNet RGB stats
train_transform = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.05)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.RandomRotation(degrees=5)], p=0.2),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.05, contrast=0.05)], p=0.15),
    transforms.RandomApply([transforms.RandomAffine(degrees=0, translate=(0.02, 0.02))], p=0.15),
    transforms.ToTensor(),
    transforms.Normalize(VIT_MEAN, VIT_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(VIT_MEAN, VIT_STD),
])

train_ds = DRDataset(train_df, TRAIN_DIR, transform=train_transform)
val_ds   = DRDataset(val_df,   VAL_DIR,   transform=val_transform)
test_ds  = DRDataset(test_df,  TEST_LABELS_DIR, transform=val_transform)

# Base loaders — train loader is built later depending on USE_FOCAL_LOSS
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

print(f'Val  loader : {len(val_loader)} batches/epoch')
print(f'Test loader : {len(test_loader)} batches/epoch')


In [ ]:
train_labels     = train_df['level'].values
class_weights_np = compute_class_weight(
    class_weight='balanced', classes=np.arange(NUM_CLASSES), y=train_labels,
)
class_weights_t = torch.tensor(class_weights_np, dtype=torch.float32).to(device)

# Per-sample weights mirror the class weights — used by WeightedRandomSampler
sample_weights  = torch.as_tensor(class_weights_np[train_labels], dtype=torch.double)
weighted_sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True,
)

print('Class weights (balanced inverse-freq):')
for name, w in zip(CLASS_NAMES, class_weights_np):
    print(f'  {name:<18}: {w:.4f}')


### FocalLoss

Focal loss down-weights easy, well-classified examples via the `(1 − pₜ)^γ` term. Pass `alpha=class_weights_t` to also correct for class imbalance via loss scaling, or `alpha=None` to use it alongside WRS.

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal loss with optional per-class alpha weighting.
    alpha=None → gamma focusing only (use alongside WRS for balance).
    alpha=Tensor → also corrects for class imbalance via loss scaling.
    """
    def __init__(self, alpha=None, gamma: float = 2.0):
        super().__init__()
        if alpha is not None:
            self.register_buffer('alpha', alpha)
        else:
            self.alpha = None
        self.gamma = gamma

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return ((1.0 - pt) ** self.gamma * ce).mean()


print('FocalLoss defined.')


## 5. Model

`vit_base_patch16_224.augreg_in21k_ft_in1k` — pre-trained on ImageNet-21k, fine-tuned on ImageNet-1k.  
The 1000-class head is replaced with a 5-class linear layer. This cell creates the global `model`
used by the training and eval utilities.

### Run-epoch utilities

`run_epoch_clipped` extends `run_epoch` with gradient clipping — used in both training phases.

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model      : {MODEL_NAME}')
print(f'Parameters : {total_params:,} total')
print(f'Head       : {model.head}')


## 6. Training Utilities

`run_epoch` — standard forward/backward pass.  
`run_epoch_clipped` — same but with optional gradient clipping (used in both training phases).  
Both functions operate on the global `model`, `criterion`, `optimizer`, `scaler` bindings.

In [ ]:
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)


def run_epoch(loader, train_mode: bool):
    model.train(train_mode)
    total_loss, total = 0.0, 0
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, leave=False, desc='train' if train_mode else 'val'):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type='cuda', enabled=use_amp):
            logits = model(images)
            loss   = criterion(logits, labels)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item() * labels.size(0)
        total      += labels.size(0)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return total_loss / total, all_labels, all_preds


def run_epoch_clipped(loader, train_mode: bool, grad_clip: float = 0.0):
    model.train(train_mode)
    total_loss, total = 0.0, 0
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, leave=False, desc='train' if train_mode else 'eval'):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type='cuda', enabled=use_amp):
            logits = model(images)
            loss   = criterion(logits, labels)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item() * labels.size(0)
        total      += labels.size(0)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return total_loss / total, all_labels, all_preds


print('run_epoch and run_epoch_clipped defined.')


## 7. Experiment 8 — Two-Phase Training: FocalLoss vs WRS

Two-phase fine-tuning with a single boolean to switch between FocalLoss and WRS for class imbalance.

| | `USE_FOCAL_LOSS = True` | `USE_FOCAL_LOSS = False` |
|---|---|---|
| Loss | `FocalLoss(alpha=class_weights, γ=2)` | `CrossEntropyLoss` |
| Sampler | Standard shuffle | `WeightedRandomSampler` |
| Imbalance correction | via loss scaling | via batch resampling |

**Training protocol** (identical to Exp3):  
Phase 1 → freeze backbone, train head only.  
Phase 2 → unfreeze all layers, fine-tune at a lower LR with gradient clipping.

### Loader Setup

Builds the train loader based on `USE_FOCAL_LOSS`. Prints a config summary so you can verify the run settings at a glance.

In [ ]:
# Build the train loader — depends on USE_FOCAL_LOSS
if USE_FOCAL_LOSS:
    # Plain shuffle — FocalLoss alpha handles the imbalance directly in the loss
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
        persistent_workers=NUM_WORKERS > 0,
    )
    _balance_method = 'FocalLoss(alpha=class_weights, gamma=2)'
else:
    # WRS rebalances batches; loss stays plain CrossEntropyLoss
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, sampler=weighted_sampler,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
        persistent_workers=NUM_WORKERS > 0,
    )
    _balance_method = 'CrossEntropyLoss + WeightedRandomSampler'

_sched_p1 = 'LinearWarmup+CosineDecay' if USE_WARMUP_P1 else 'CosineAnnealingLR'
_sched_p2 = 'LinearWarmup+CosineDecay' if USE_WARMUP_P2 else 'CosineAnnealingLR'
_warmup_note = f'  (warmup={WARMUP_EPOCHS} epochs)'

print(f'Imbalance method  : {_balance_method}')
print(f'Phase 1 LR/Epochs : {PHASE1_LR:.0e} / {PHASE1_EPOCHS}')
print(f'Phase 2 LR/Epochs : {PHASE2_LR:.0e} / {PHASE2_EPOCHS}')
print(f'Batch / WD / Clip : {BATCH_SIZE} / {WEIGHT_DECAY} / {GRAD_CLIP}')
print(f'Patience          : {PATIENCE}')
print(f'Scheduler P1      : {_sched_p1}' + (_warmup_note if USE_WARMUP_P1 else ''))
print(f'Scheduler P2      : {_sched_p2}' + (_warmup_note if USE_WARMUP_P2 else ''))
print(f'Train loader      : {len(train_loader)} batches/epoch')


### Model, Criterion & Output Dirs

Fresh pretrained model, criterion chosen by `USE_FOCAL_LOSS`, checkpoint and figure directories created.

In [ ]:
set_seed(SEED)
vit_model  = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES).to(device)
vit_scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

if USE_FOCAL_LOSS:
    vit_criterion = FocalLoss(alpha=class_weights_t, gamma=2.0)
else:
    vit_criterion = nn.CrossEntropyLoss()

FINAL_OUT_DIR  = ROOT / 'outputs' / 'vit_b16_final'
FINAL_CKPT_DIR = FINAL_OUT_DIR / 'checkpoints'
FINAL_FIG_DIR  = FINAL_OUT_DIR / 'figures'
for d in (FINAL_CKPT_DIR, FINAL_FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

PHASE1_CKPT = FINAL_CKPT_DIR / 'vit_b16_final_phase1_best.pt'
FINAL_CKPT        = FINAL_CKPT_DIR / 'vit_b16_final_best.pt'

print(f'Criterion  : {vit_criterion}')
print(f'Checkpoint → {FINAL_CKPT_DIR}')


### Phase 1 — Head Only (Frozen Backbone)

All layers except the classification head are frozen. Training only the head first prevents early distortion of the pre-trained features.

In [ ]:
# Freeze everything except the classification head
for name, param in vit_model.named_parameters():
    if 'head' not in name:
        param.requires_grad = False

trainable_p1 = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)
print(f'Phase 1: {trainable_p1:,} trainable parameters (head only)\n')

# Stash any existing globals so we can restore them after training
_stash_model     = globals().get('model',     None)
_stash_criterion = globals().get('criterion', None)
_stash_optimizer = globals().get('optimizer', None)
_stash_scaler    = globals().get('scaler',    None)

model     = vit_model
criterion = vit_criterion
scaler    = vit_scaler
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=PHASE1_LR, weight_decay=WEIGHT_DECAY,
)

if USE_WARMUP_P1:
    _p1_w = max(1, min(WARMUP_EPOCHS, PHASE1_EPOCHS - 1))
    scheduler_p1 = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[
        torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1e-2, end_factor=1.0, total_iters=_p1_w),
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, PHASE1_EPOCHS - _p1_w)),
    ], milestones=[_p1_w])
else:
    scheduler_p1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS)

vit_history = {
    'phase1_train': [], 'phase1_val': [], 'phase1_train_acc': [], 'phase1_val_acc': [], 'phase1_val_kappa': [],
    'phase2_train': [], 'phase2_val': [], 'phase2_train_acc': [], 'phase2_val_acc': [], 'phase2_val_kappa': [],
}
p1_best_val, p1_best_epoch, p1_patience = float('inf'), 0, PATIENCE

print(f"{'Epoch':<8} {'Tr Loss':>10} {'Tr Acc':>8} {'Val Loss':>10} {'Val Acc':>8} {'Val QWK':>9}")
print('─' * 60)

for epoch in range(1, PHASE1_EPOCHS + 1):
    tr_loss, tr_labels, tr_preds = run_epoch_clipped(train_loader, train_mode=True,  grad_clip=0.0)
    vl_loss, vl_labels, vl_preds = run_epoch_clipped(val_loader,       train_mode=False)
    scheduler_p1.step()

    tr_acc   = accuracy_score(tr_labels, tr_preds)
    vl_acc   = accuracy_score(vl_labels, vl_preds)
    vl_kappa = cohen_kappa_score(vl_labels, vl_preds, weights='quadratic')
    vit_history['phase1_train'].append(tr_loss)
    vit_history['phase1_val'].append(vl_loss)
    vit_history['phase1_train_acc'].append(tr_acc)
    vit_history['phase1_val_acc'].append(vl_acc)
    vit_history['phase1_val_kappa'].append(vl_kappa)

    marker = ''
    if vl_loss < p1_best_val:
        p1_best_val, p1_best_epoch, p1_patience = vl_loss, epoch, PATIENCE
        torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'val_loss': vl_loss},
                   PHASE1_CKPT)
        marker = '  ← best'
    else:
        p1_patience -= 1

    print(f'{epoch:<8} {tr_loss:>10.4f} {tr_acc:>8.4f} {vl_loss:>10.4f} {vl_acc:>8.4f} {vl_kappa:>9.4f}{marker}')
    if p1_patience == 0:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print(f'\nPhase 1 done. Best val loss: {p1_best_val:.4f}  (epoch {p1_best_epoch})')


### Phase 2 — Full Fine-Tune (All Layers)

Reload Phase 1's best checkpoint, unfreeze everything, and fine-tune with a lower LR and gradient clipping.

In [ ]:
# Unfreeze everything and reload Phase 1's best weights
for param in vit_model.parameters():
    param.requires_grad = True

ckpt_p1 = torch.load(PHASE1_CKPT, map_location=device, weights_only=False)
model.load_state_dict(ckpt_p1['model_state'])
print(f'Phase 2: loaded Phase 1 best (epoch {ckpt_p1["epoch"]}, val_loss={ckpt_p1["val_loss"]:.4f})\n')

optimizer = torch.optim.AdamW(model.parameters(), lr=PHASE2_LR, weight_decay=WEIGHT_DECAY)

if USE_WARMUP_P2:
    _p2_w = max(1, min(WARMUP_EPOCHS, PHASE2_EPOCHS - 1))
    scheduler_p2 = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[
        torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1e-2, end_factor=1.0, total_iters=_p2_w),
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, PHASE2_EPOCHS - _p2_w)),
    ], milestones=[_p2_w])
else:
    scheduler_p2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS)

p2_best_val, p2_best_epoch, p2_patience = float('inf'), 0, PATIENCE

print(f"{'Epoch':<8} {'Tr Loss':>10} {'Tr Acc':>8} {'Val Loss':>10} {'Val Acc':>8} {'Val QWK':>9}")
print('─' * 60)

for epoch in range(1, PHASE2_EPOCHS + 1):
    tr_loss, tr_labels, tr_preds = run_epoch_clipped(train_loader, train_mode=True,  grad_clip=GRAD_CLIP)
    vl_loss, vl_labels, vl_preds = run_epoch_clipped(val_loader,       train_mode=False)
    scheduler_p2.step()

    tr_acc   = accuracy_score(tr_labels, tr_preds)
    vl_acc   = accuracy_score(vl_labels, vl_preds)
    vl_kappa = cohen_kappa_score(vl_labels, vl_preds, weights='quadratic')
    vit_history['phase2_train'].append(tr_loss)
    vit_history['phase2_val'].append(vl_loss)
    vit_history['phase2_train_acc'].append(tr_acc)
    vit_history['phase2_val_acc'].append(vl_acc)
    vit_history['phase2_val_kappa'].append(vl_kappa)

    marker = ''
    if vl_loss < p2_best_val:
        p2_best_val, p2_best_epoch, p2_patience = vl_loss, epoch, PATIENCE
        torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'val_loss': vl_loss},
                   FINAL_CKPT)
        marker = '  ← best'
    else:
        p2_patience -= 1

    print(f'{epoch:<8} {tr_loss:>10.4f} {tr_acc:>8.4f} {vl_loss:>10.4f} {vl_acc:>8.4f} {vl_kappa:>9.4f}{marker}')
    if p2_patience == 0:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print(f'\nPhase 2 done. Best val loss: {p2_best_val:.4f}  (epoch {p2_best_epoch})')
print(f'Checkpoint → {FINAL_CKPT}')

# Restore original globals
if _stash_model is not None:
    model, criterion, optimizer, scaler = (
        _stash_model, _stash_criterion, _stash_optimizer, _stash_scaler,
    )


### Training Curves

Combined loss plot for both phases with a dashed line marking the Phase 1 → Phase 2 boundary.

In [ ]:
p2_start  = len(vit_history['phase1_train'])
all_train = vit_history['phase1_train'] + vit_history['phase2_train']
all_val   = vit_history['phase1_val']   + vit_history['phase2_val']
all_kappa = vit_history['phase1_val_kappa'] + vit_history['phase2_val_kappa']
epochs_x  = range(1, len(all_train) + 1)

_title_m = 'FocalLoss(alpha=class_weights, γ=2)' if USE_FOCAL_LOSS else 'CrossEntropyLoss + WRS'

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Loss curves
ax1.plot(epochs_x, all_train, label='Train Loss', marker='o', markersize=4)
ax1.plot(epochs_x, all_val,   label='Val Loss',   marker='s', markersize=4)
if 0 < p2_start < len(all_train):
    ax1.axvline(p2_start + 0.5, color='gray', linestyle='--', alpha=0.7, label='Phase 1 → Phase 2')
ax1.set_title(f'Final Model — Train vs Val Loss  |  {_title_m}', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax1.legend()
ax1.spines[['top', 'right']].set_visible(False)

# Validation QWK curve
ax2.plot(epochs_x, all_kappa, label='Val QWK', color='tab:green', marker='D', markersize=4)
if 0 < p2_start < len(all_kappa):
    ax2.axvline(p2_start + 0.5, color='gray', linestyle='--', alpha=0.7, label='Phase 1 → Phase 2')
ax2.set_title('Final Model — Validation QWK per Epoch', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Quadratic Weighted Kappa')
ax2.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax2.legend()
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
fig.savefig(FINAL_FIG_DIR / 'vit_b16_final_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

### Evaluation & Results Logging

Loads the best Phase 2 checkpoint and runs inference on the held-out test set. Results are appended to the CSV log — re-running adds a new row without overwriting.

In [ ]:
# Independent of training — only needs cells 1-6 + checkpoint paths to have been run.
_prev_model, _prev_criterion = globals().get('model'), globals().get('criterion')

_ckpt_path = ROOT / 'outputs' / 'vit_b16_final' / 'checkpoints' / 'vit_b16_final_best.pt'
model     = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
ckpt      = torch.load(_ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded {_ckpt_path.name}  (epoch {ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.4f})')

_, y_true, y_pred = run_epoch(test_loader, train_mode=False)

final_qwk      = cohen_kappa_score(y_true, y_pred, weights='quadratic')
final_acc      = accuracy_score(y_true, y_pred)
final_macro_f1 = f1_score(y_true, y_pred, average='macro')
final_f1_per   = f1_score(y_true, y_pred, average=None, labels=list(range(5)))

print('\nClassification Report (Test Set):')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))
print(f'QWK      : {final_qwk:.4f}')
print(f'Accuracy : {final_acc:.4f}')
print(f'Macro F1 : {final_macro_f1:.4f}')

# Append one row per run — never overwrites previous results
LOG_PATH = ROOT / 'outputs' / 'vit_b16_final' / 'results_log.csv'
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

_use_fl  = globals().get('USE_FOCAL_LOSS', None)
_bmethod = ('FocalLoss(alpha=class_weights,gamma=2)' if _use_fl is True
             else 'CrossEntropyLoss+WRS' if _use_fl is False else 'unknown')

log_row = {
    'timestamp':               datetime.now().isoformat(),
    'use_focal_loss':          _use_fl,
    'balance_method':          _bmethod,
    'phase1_epochs':           globals().get('PHASE1_EPOCHS'),
    'phase2_epochs':           globals().get('PHASE2_EPOCHS'),
    'phase1_lr':               globals().get('PHASE1_LR'),
    'phase2_lr':               globals().get('PHASE2_LR'),
    'batch_size':              globals().get('BATCH_SIZE'),
    'weight_decay':            globals().get('WEIGHT_DECAY'),
    'grad_clip':               globals().get('GRAD_CLIP'),
    'scheduler_p1':            ('LinearWarmup+CosineDecay' if globals().get('USE_WARMUP_P1')
                               else 'CosineAnnealingLR'),
    'scheduler_p2':            ('LinearWarmup+CosineDecay' if globals().get('USE_WARMUP_P2')
                               else 'CosineAnnealingLR'),
    'warmup_epochs':           globals().get('WARMUP_EPOCHS', 0),
    'early_stopping_patience': globals().get('PATIENCE'),
    'p1_best_epoch':           globals().get('p1_best_epoch'),
    'p1_best_val_loss':        globals().get('p1_best_val'),
    'p2_best_epoch':           globals().get('p2_best_epoch'),
    'p2_best_val_loss':        globals().get('p2_best_val'),
    'test_qwk':                round(final_qwk, 4),
    'test_accuracy':           round(final_acc, 4),
    'test_macro_f1':           round(final_macro_f1, 4),
    'f1_no_dr':                round(float(final_f1_per[0]), 4),
    'f1_mild':                 round(float(final_f1_per[1]), 4),
    'f1_moderate':             round(float(final_f1_per[2]), 4),
    'f1_severe':               round(float(final_f1_per[3]), 4),
    'f1_proliferative':        round(float(final_f1_per[4]), 4),
}
write_header = not LOG_PATH.exists()
pd.DataFrame([log_row]).to_csv(LOG_PATH, mode='a', header=write_header, index=False)
print(f'\nLogged → {LOG_PATH}')

if _prev_model is not None:
    model = _prev_model; criterion = _prev_criterion


### Confusion Matrices

Raw counts (left) and row-normalised recall (right) shown side by side.

In [ ]:
cm_final      = confusion_matrix(y_true, y_pred)
cm_final_norm = cm_final.astype(float) / cm_final.sum(axis=1, keepdims=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Raw counts
im1 = ax1.imshow(cm_final, interpolation='nearest', cmap='Blues')
fig.colorbar(im1, ax=ax1)
ax1.set_xticks(range(NUM_CLASSES)); ax1.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax1.set_yticks(range(NUM_CLASSES)); ax1.set_yticklabels(CLASS_NAMES)
ax1.set_xlabel('Predicted'); ax1.set_ylabel('True')
ax1.set_title('Confusion Matrix — Raw Counts', fontweight='bold')
thresh1 = cm_final.max() / 2
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax1.text(j, i, f'{cm_final[i,j]:,}', ha='center', va='center', fontsize=9,
                 color='white' if cm_final[i,j] > thresh1 else 'black')

# Normalised by true class (row-normalised recall)
im2 = ax2.imshow(cm_final_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
fig.colorbar(im2, ax=ax2, format='%.2f')
ax2.set_xticks(range(NUM_CLASSES)); ax2.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax2.set_yticks(range(NUM_CLASSES)); ax2.set_yticklabels(CLASS_NAMES)
ax2.set_xlabel('Predicted'); ax2.set_ylabel('True')
ax2.set_title('Confusion Matrix — Normalised (Row = True Class)', fontweight='bold')
thresh2 = 0.5
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax2.text(j, i, f'{cm_final_norm[i,j]:.2f}', ha='center', va='center', fontsize=9,
                 color='white' if cm_final_norm[i,j] > thresh2 else 'black')

plt.suptitle('Final Model — Test Set Confusion Matrices', fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(FINAL_FIG_DIR / 'vit_b16_final_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### Results Summary

In [ ]:
nan = float('nan')
_g  = globals()

def _v(key):
    v = _g.get(key)
    return float(v) if v is not None else nan

def _fi(key, i):
    arr = _g.get(key)
    if arr is None: return nan
    try: return float(arr[i])
    except: return nan

def fmt(v, w=8):
    return f'{"—":>{w}}' if v != v else f'{v:>{w}.4f}'

W = 50
print('=' * W)
print(f'  {"Metric":<22} {"Final":>8}')
print('=' * W)
for label, val in [
    ('Test QWK',      final_qwk),
    ('Test Accuracy', final_acc),
    ('Test Macro F1', final_macro_f1),
]:
    print(f'  {label:<22}  {fmt(val)}')
print()
print('  Per-class F1:')
for i, name in enumerate(CLASS_NAMES):
    print(f'    {name:<20}  {fmt(float(final_f1_per[i]))}')
print('=' * W)
_use_fl = globals().get('USE_FOCAL_LOSS', None)
if _use_fl is not None:
    _m = 'FocalLoss(alpha=class_weights, gamma=2)' if _use_fl else 'CrossEntropyLoss + WRS'
    print(f'  Balance method: {_m}')
print()
